1. Smart Customer Support Ticket Router
### How it works:
* An input node takes a customer message.
* A conditional node reads the message.
* If it is about money, it goes to the Billing Agent.
* If it is about a bug, it goes to the Tech Support Agent.
* If it is angry, it goes straight to a Human Handoff node

                    ┌─────────────────┐
                    │      START      │
                    └────────┬────────┘
                             │
                             ▼
                 ┌──────────────────────┐
                 │  classify_intent     │
                 │                      │
                 │ Billing?             │
                 │ Technical?           │
                 │ Human handoff?       │
                 │ General?             │
                 └──────────┬───────────┘
                            │
                    conditional routing
                            │
          ┌─────────────────┼──────────────────┬─────────────────┐
          ▼                 ▼                  ▼                 ▼
   ┌─────────────┐   ┌───────────────┐  ┌───────────────┐  ┌─────────────┐
   │   Billing   │   │ TechnicalSup  │  │ HumanHandOff  │  │ general_node│
   └──────┬──────┘   └───────┬───────┘  └───────┬───────┘  └──────┬──────┘
          │                  │                   │                 │
          └──────────────────┴───────────────────┴─────────────────┘
                                     │
                                     ▼
                              ┌─────────────┐
                              │     END     │
                              └─────────────┘


In [ ]:
from typing import TypedDict, Optional, Literal
from langgraph.graph import StateGraph, START, END



# ============================================================
# 1. Define the shared state
# ============================================================

class CustomerSupportState(TypedDict):
    user_message: str
    intent: list[str]
    human_handoff: bool
    response: Optional[str]


# ============================================================
# 2. Classify the customer's intent
# ============================================================

def classify_intent(state: CustomerSupportState) -> CustomerSupportState:

    user_message = state["user_message"].lower().strip()

    if any(word in user_message for word in [
        "refund",
        "return",
        "money back",
        "chargeback"
    ]):
        state["intent"] = ["billing"]

    elif any(word in user_message for word in [
        "bug",
        "error",
        "not working",
        "technical",
        "crash"
    ]):
        state["intent"] = ["technical_support"]

    elif any(word in user_message for word in [
        "angry",
        "fuck",
        "bad",
        "frustrated",
        "hate"
    ]):
        state["intent"] = ["human_handoff"]

    else:
        state["intent"] = ["general_support"]   

    return state


# ============================================================
# 3. Decide which node to visit
# ============================================================

def route_intent(state: CustomerSupportState) -> Literal["billing","technical_support","human_handoff","general_support"]:

    intent = state["intent"]

    if intent == "billing":
        return "billing"

    elif intent == "technical_support":
        return "technical_support"

    elif intent == "human_handoff":
        return "human_handoff"

    else:
        return "general_support"


# ============================================================
# 4. Billing node
# ============================================================

def billing(state: CustomerSupportState) -> CustomerSupportState:

    state["response"] = (
        "I will help you with your refund request."
    )

    return state


# ============================================================
# 5. Technical support node
# ============================================================

def technical_support(state: CustomerSupportState) -> CustomerSupportState:

    state["response"] = ("I will help you with your Technical Support request.")

    return state


# ============================================================
# 6. Human handoff node
# ============================================================

def human_handoff(state: CustomerSupportState) -> CustomerSupportState:

    state["human_handoff"] = True

    state["response"] = ("I'm transferring you to a human support agent.")

    return state


# ============================================================
# 7. General support node
# ============================================================

def general_support(state: CustomerSupportState) -> CustomerSupportState:


    state["response"] = ("How can I help you today?")

    return state


# ============================================================
# 8. Build the graph
# ============================================================

graph = StateGraph(CustomerSupportState)


# Add nodes
graph.add_node("classify_intent", classify_intent)
graph.add_node("billing", billing)
graph.add_node("technical_support", technical_support)
graph.add_node("human_handoff", human_handoff)
graph.add_node("general_support", general_support)


# START → classify_intent
graph.add_edge(
    START,
    "classify_intent"
)


# classify_intent → one of the four branches
graph.add_conditional_edges(
    "classify_intent",
    route_intent
)


# Each branch → END
graph.add_edge("billing", END)
graph.add_edge("technical_support", END)
graph.add_edge("human_handoff", END)
graph.add_edge("general_support", END)


# Compile
workflow = graph.compile()


# ============================================================
# 9. Run the workflow
# ============================================================

initial_state: CustomerSupportState = {
    "user_message": "I need help with my refund.",
    "intent": "",
    "human_handoff": False,
    "response": None,
}


final_state = workflow.invoke(initial_state)


print(final_state)

{'user_message': 'I need help with my refund.', 'intent': ['billing'], 'human_handoff': False, 'response': 'How can I help you today?'}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
